In [3]:
import pandas as pd
import numpy as np
from datetime import datetime
import os
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [4]:
base_url = 'https://raw.githubusercontent.com/londonaicentre/datatakehome/refs/heads/main/data'
encounters = pd.read_csv(os.path.join(base_url,'encounters.csv'))
encounters_diff = pd.read_csv(os.path.join(base_url,'encounters_schema_change_batch.csv'))

In [30]:
print('encounter.csv column headers:')
print(sorted([*encounters.columns]))
print("")
print('encounter_schema_change_batch.csv column headers:')
print(sorted([*encounters_diff.columns]))

encounter.csv column headers:
['BASE_ENCOUNTER_COST', 'CODE', 'DESCRIPTION', 'ENCOUNTERCLASS', 'Id', 'ORGANIZATION', 'PATIENT', 'PAYER', 'PAYER_COVERAGE', 'PROVIDER', 'REASONCODE', 'REASONDESCRIPTION', 'START', 'STOP', 'TOTAL_CLAIM_COST']

encounter_schema_change_batch.csv column headers:
['BASE_ENCOUNTER_COST', 'CODE', 'DESCRIPTION', 'ENCOUNTER_TYPE', 'Id', 'ORGANIZATION', 'PATIENT', 'PAYER', 'PAYER_COVERAGE', 'PROVIDER', 'REASONCODE', 'REASONDESCRIPTION', 'SOURCE_SYSTEM', 'START', 'STOP', 'TOTAL_CLAIM_COST']


In [ ]:
# check encounter type unique values
print(sorted((encounters['ENCOUNTERCLASS'].unique())))
print(sorted(encounters_diff['ENCOUNTER_TYPE'].unique()))

['ambulatory', 'emergency', 'inpatient', 'outpatient', 'urgentcare', 'wellness']
['ambulatory', 'emergency', 'inpatient', 'outpatient', 'urgentcare', 'wellness']


In [8]:
encounters.head(2)

,Id,START,STOP,PATIENT,ORGANIZATION,PROVIDER,PAYER,ENCOUNTERCLASS,CODE,DESCRIPTION,BASE_ENCOUNTER_COST,TOTAL_CLAIM_COST,PAYER_COVERAGE,REASONCODE,REASONDESCRIPTION
0,d0c40d10-8d87-447e-836e-99d26ad52ea5,2010-01-23T17:45:28Z,2010-01-23T18:10:28Z,034e9e3b-2def-4559-bb2a-7850888ae060,e002090d-4e92-300e-b41e-7d1f21dee4c6,e6283e46-fd81-3611-9459-0edb1c3da357,6e2f1a2d-27bd-3701-8d08-dae202c58632,ambulatory,185345009,Encounter for symptom,129.16,129.16,54.16,10509002.0,Acute bronchitis (disorder)
1,e88bc3a9-007c-405e-aabc-792a38f4aa2b,2012-01-23T17:45:28Z,2012-01-23T18:00:28Z,034e9e3b-2def-4559-bb2a-7850888ae060,772ee193-bb9f-30eb-9939-21e86c8e4da5,6f1d59a7-a5bd-3cf9-9671-5bad2f351c28,6e2f1a2d-27bd-3701-8d08-dae202c58632,wellness,162673000,General examination of patient (procedure),129.16,129.16,129.16,NaN,NaN


In [9]:
encounters_diff.head(2)

,Id,PATIENT,ENCOUNTER_TYPE,CODE,DESCRIPTION,START,STOP,ORGANIZATION,PROVIDER,PAYER,BASE_ENCOUNTER_COST,TOTAL_CLAIM_COST,PAYER_COVERAGE,REASONCODE,REASONDESCRIPTION,SOURCE_SYSTEM
0,509c21e2-80e9-4fc7-8279-3137abd0eff6,941f66eb-5ca6-45cc-b205-6de1f1a59233,inpatient,185347001,Encounter for problem,1532389366000,1.532487e+12,49318f80-bd8b-3fc7-a096-ac43088b0c12,680f4af2-775d-34c4-b213-28791e0813e3,6e2f1a2d-27bd-3701-8d08-dae202c58632,129.16,129.16,54.16,NaN,NaN,EPIC_PROD
1,450ed060-fbf6-4530-92a6-5ba1c071f95a,e4945ad4-5f21-460f-a420-98a32ef61d3d,wellness,162673000,General examination of patient (procedure),854303121000,8.543049e+11,79e14b44-8026-3599-b8ac-14813879825e,48785c4b-b4fb-3e03-9da2-1c707ea528e0,5059a55e-5d6e-34d1-b6cb-d83d16e57bcf,129.16,129.16,59.16,NaN,NaN,CERNER_LEGACY


1. First perform data modelling to compare difference in schema.
    Difference:
    - column name: encounterclass changed from encounterclass to encounter_type, the unique values of both columns are the same
    - data type: start and stop columns changed from utc timestamp to unix seconds timestamp
    - column headers: extra column present to indicate source systems
2. Convert start and stop columns to timestamp format. However, it was unknown whether the timestamp was in UTC format or not, this would need to be confirmed with database documentation if available. 
3. Convert column name ENCOUNTER_TYPE to ENCOUNTERCLASS

3. The approach depends on whether the systems share the same indexing system for generation of uuids (encounter, patient, origaniztion, payer)
    - if the uuids share the same master index, then the approach would be to join the 2 encounter tables by id and patient, to identify records not present in the encounter, then append to the encounter table. Replace any missing data from the encounters table with data from the schema_change table. However, any inconsistent data would need further investigation.
    - if the uuids do not share the same master index, then it was not impossible to conclude whether the encounters in the schema_change table corresponds to that in the encounters table. (even if the uuids are the same) To unify the 2 tables, I would add a suffix to the encounters in the schema_change table using the source_system column. Then I would append the data to the encounters table. 